# Edge-Swap Reservoir Computing Experiment

**Biologically-realistic** implementation with two key fixes that make
perturbations actually visible in t_div:

1. **Fixed spectral scaling** — the normalization factor `J/ρ(G)` is computed
   once from the *original* unperturbed graph and applied to every perturbed
   copy. Without this fix, the reservoir re-normalises itself after every
   perturbation and the damage is invisible.

2. **Sparse input weights** — only ~20 % of nodes receive the Lorenz drive,
   mimicking specialised sensory regions in real connectomes.

3. **Edge-swap integrity** — swapped edges are always re-inserted (with up to
   100 candidate retries) so total edge count is never reduced accidentally.

> No external `.py` helper files needed — everything is inlined.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import time
from pathlib import Path
from scipy import stats


## Core functions

In [ ]:
# ── Lorenz attractor (Euler integration) ─────────────────────────────────────

def lorenz_euler(n_steps, dt=0.01, sigma=10.0, rho=28.0, beta=8/3, state0=None):
    """Integrate the Lorenz system with the Euler method."""
    data  = np.empty((n_steps, 3))
    state = np.array([1.0, 1.0, 1.0]) if state0 is None else np.array(state0, dtype=float)
    for i in range(n_steps):
        x, y, z = state
        state = state + dt * np.array([
            sigma * (y - x),
            x * (rho - z) - y,
            x * y - beta * z,
        ])
        data[i] = state
    return data, state


In [ ]:
# ── Divergence-time metric ───────────────────────────────────────────────────

def t_div(Y_pred, Y_true, epsilon=1.0, dt=0.01):
    """First time Euclidean distance between predicted and true trajectory
    exceeds epsilon (returned in seconds)."""
    distances = np.linalg.norm(Y_pred - Y_true, axis=1)
    exceeds   = np.where(distances > epsilon)[0]
    if len(exceeds) == 0:
        return len(Y_pred) * dt      # never diverges within window
    return exceeds[0] * dt


In [ ]:
# ── Graph perturbations ───────────────────────────────────────────────────────

def perturb_graph(G, mode, p, rng):
    """Return a perturbed copy of G.

    Parameters
    ----------
    mode : 'drop_nodes' | 'drop_edges' | 'swap_edges' | 'swap_nodes'
    p    : perturbation fraction in [0, 1]
    rng  : numpy.random.Generator
    """
    H = G.copy()

    if mode == "drop_nodes":
        nodes   = list(H.nodes())
        k       = max(0, round(p * len(nodes)))
        to_drop = rng.choice(nodes, size=k, replace=False).tolist()
        H.remove_nodes_from(to_drop)

    elif mode == "drop_edges":
        edges = list(H.edges())
        k     = max(0, round(p * len(edges)))
        idxs  = rng.choice(len(edges), size=k, replace=False)
        H.remove_edges_from([edges[i] for i in idxs])

    elif mode == "swap_edges":
        # Rewire edges while preserving total edge count.
        # If no valid target is found in 100 attempts the original edge
        # is restored — graph density is never reduced.
        edges = list(H.edges())
        nodes = list(H.nodes())
        k     = max(0, round(p * len(edges)))
        for i in rng.choice(len(edges), size=k, replace=False):
            u, v  = edges[i]
            attrs = H[u][v].copy()
            H.remove_edge(u, v)
            swapped = False
            for _ in range(100):
                new_v = rng.choice(nodes)
                if not H.has_edge(u, new_v) and u != new_v:
                    H.add_edge(u, new_v, **attrs)
                    swapped = True
                    break
            if not swapped:
                H.add_edge(u, v, **attrs)   # restore original

    elif mode == "swap_nodes":
        nodes   = list(H.nodes())
        k       = max(0, round(p * len(nodes)) // 2 * 2)   # must be even
        chosen  = rng.choice(nodes, size=k, replace=False)
        mapping = dict(zip(chosen, np.roll(chosen, k // 2)))
        H = nx.relabel_nodes(H, mapping)

    else:
        raise ValueError(f"Unknown perturbation mode: {mode!r}")

    return H


In [ ]:
# ── Build adjacency/weight matrix D from a NetworkX graph ────────────────────

def graph_to_D(G):
    """Return the (N x N) symmetric weight matrix D for graph G.
    Edge weight = number_of_fibers attribute (default 1.0)."""
    N         = G.number_of_nodes()
    node_ids  = sorted(G.nodes())
    id_to_idx = {nid: i for i, nid in enumerate(node_ids)}
    D = np.zeros((N, N))
    for u, v, attrs in G.edges(data=True):
        if u not in id_to_idx or v not in id_to_idx:
            continue
        i, j = id_to_idx[u], id_to_idx[v]
        w = float(attrs.get("number_of_fibers", 1.0))
        D[i, j] = w
        D[j, i] = w
    return D


In [ ]:
# ── Single reservoir run ─────────────────────────────────────────────────────
#
# KEY BIOLOGICAL FIX:
#   scaling_factor is computed from the ORIGINAL graph and passed in.
#   The perturbed graph is NOT re-normalised → structural damage actually
#   changes the effective spectral radius and therefore the reservoir dynamics.
#
# SECOND FIX (sparse W_in):
#   Only ~20 % of nodes receive the external Lorenz input, mimicking
#   specialised sensory regions in real connectomes.

def run_one(G, seed, scaling_factor,
            lambda_reg=3e-6,
            t_thermalization=30, t_training=200, t_prediction=50,
            sigma=10.0, rho=28.0, beta=8/3,
            dt=0.01, input_scale=0.5, alpha=0.3,
            sparse_input_frac=0.2):
    """Train an ESN on graph G and run autonomous prediction.

    Parameters
    ----------
    scaling_factor : float
        J / rho(D_original) — computed from the UNPERTURBED graph.
    sparse_input_frac : float
        Fraction of nodes that receive the Lorenz input (default 20 %).

    Returns (Y_pred, Y_true) or (None, None) if graph is degenerate.
    """
    N = G.number_of_nodes()
    if N == 0:
        return None, None

    ws = round(t_thermalization / dt)
    ts = round(t_training      / dt)
    ns = round(t_prediction    / dt)

    # Build W using the FIXED scaling factor from the original graph
    D = graph_to_D(G)
    W = scaling_factor * D

    # Lorenz driver (normalised)
    lf, ll = lorenz_euler(ws + ts, dt=dt, sigma=sigma, rho=rho, beta=beta)
    u_mean = lf.mean(axis=0)
    u_std  = lf.std(axis=0);  u_std[u_std == 0] = 1.0
    ln     = (lf - u_mean) / u_std
    u_tr   = ln[ws:]

    # Sparse input weights: only sparse_input_frac of nodes get input
    rng_in = np.random.default_rng(seed)
    W_in   = input_scale * rng_in.normal(size=(N, 3))
    mask   = rng_in.choice([0.0, 1.0], size=(N, 1),
                            p=[1 - sparse_input_frac, sparse_input_frac])
    W_in   = W_in * mask      # zero out ~80 % of rows

    # Washout
    r = np.zeros(N)
    for t in range(ws):
        r = (1-alpha) * r + alpha * np.tanh(W @ r + W_in @ ln[t])

    # Training
    R = np.empty((ts, N))
    for t in range(ts):
        r    = (1-alpha) * r + alpha * np.tanh(W @ r + W_in @ u_tr[t])
        R[t] = r

    W_out = np.linalg.solve(
        R[:-1].T @ R[:-1] + lambda_reg * np.eye(N),
        R[:-1].T @ u_tr[1:],
    )

    # Autonomous prediction
    Y_pred = np.empty((ns, 3))
    for t in range(ns):
        u_hat_norm  = r @ W_out
        Y_pred[t]   = u_hat_norm * u_std + u_mean
        r           = (1-alpha) * r + alpha * np.tanh(W @ r + W_in @ u_hat_norm)

    # Ground-truth Lorenz continuing from end of training
    Y_true, _ = lorenz_euler(ns, dt=dt, sigma=sigma, rho=rho, beta=beta,
                              state0=ll)
    return Y_pred, Y_true


In [ ]:
# ── Full experiment ───────────────────────────────────────────────────────────

PERTURBATION_TYPES = ["drop_nodes", "drop_edges", "swap_edges", "swap_nodes"]


def run_experiment(
    file_paths,
    *,
    p_values,
    n_runs           = 5,
    epsilon          = 1.0,
    J                = 0.9,
    lambda_reg       = 3e-6,
    t_thermalization = 30,
    t_training       = 200,
    t_prediction     = 50,
    sigma            = 10.0,
    rho              = 28.0,
    beta             = 8 / 3,
    dt               = 0.01,
    input_scale      = 0.5,
    alpha            = 0.3,
    sparse_input_frac= 0.2,
    seed_base        = 42,
    perturbation_types = None,
):
    """
    Compute  E_R [ E_runs [ t_div( f_p(R) ) ] ]

    KEY: the spectral-radius scaling factor is computed once per file from the
    ORIGINAL (unperturbed) graph and is held fixed for all (mode, p, run)
    combinations of that file.  Structural damage therefore has a real effect
    on the reservoir dynamics.
    """
    if perturbation_types is None:
        perturbation_types = PERTURBATION_TYPES

    results = {
        mode: {p: [] for p in p_values}
        for mode in perturbation_types
    }

    n_files  = len(file_paths)
    n_combos = len(perturbation_types) * len(p_values) * n_files * n_runs
    done     = 0
    t_start  = time.perf_counter()

    for fi, fp in enumerate(file_paths):
        fp = Path(fp)
        G  = nx.read_graphml(fp)

        # ── Compute scaling factor from the ORIGINAL graph ────────────────
        D_orig = graph_to_D(G)
        rho_G  = np.max(np.abs(np.linalg.eigvals(D_orig)))
        if rho_G == 0:
            print(f"  Skipping {fp.name}: zero spectral radius")
            continue
        scaling_factor = J / rho_G

        for mode in perturbation_types:
            for p in p_values:
                for run_idx in range(n_runs):

                    # W_in seed varies across files & runs, fixed across p/mode
                    seed = seed_base + run_idx * 10_000 + fi

                    # Perturbation seed varies across everything
                    seed_p = (seed_base
                              + int(p * 1e6)
                              + hash(mode) % 10_000
                              + fi * 100
                              + run_idx)
                    rng_p = np.random.default_rng(seed_p)

                    H = perturb_graph(G, mode, p, rng_p)

                    t0 = time.perf_counter()
                    Y_pred, Y_true = run_one(
                        H, seed=seed,
                        scaling_factor   = scaling_factor,  # fixed!
                        lambda_reg       = lambda_reg,
                        t_thermalization = t_thermalization,
                        t_training       = t_training,
                        t_prediction     = t_prediction,
                        sigma=sigma, rho=rho, beta=beta,
                        dt=dt, input_scale=input_scale, alpha=alpha,
                        sparse_input_frac=sparse_input_frac,
                    )
                    elapsed = time.perf_counter() - t0

                    td = (0.0 if Y_pred is None
                          else t_div(Y_pred, Y_true, epsilon=epsilon, dt=dt))
                    results[mode][p].append(td)
                    done += 1

                    elapsed_total = time.perf_counter() - t_start
                    eta = elapsed_total / done * (n_combos - done)
                    print(
                        f"[{done:5d}/{n_combos}]"
                        f"  file {fi+1:3d}/{n_files}  {fp.stem}"
                        f"  {mode:12s}  p={p:.2f}  run={run_idx}"
                        f"  t_div={td:.3f}s  job={elapsed:.1f}s"
                        f"  ETA={eta/60:.1f}min",
                        flush=True,
                    )

    return results


In [ ]:
# ── Plotting ─────────────────────────────────────────────────────────────────

def plot_experiment(results, p_values, epsilon, n_runs, n_files,
                    t_training, perturbation_types=None, confidence=0.95):
    """2×2 grid: E[t_div] vs perturbation fraction p with confidence bands."""
    if perturbation_types is None:
        perturbation_types = PERTURBATION_TYPES

    fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=False)

    for ax, mode in zip(axes.flatten(), perturbation_types):
        vals  = [np.array(results[mode][p]) for p in p_values]
        means = np.array([v.mean() for v in vals])
        n_obs = np.array([len(v)   for v in vals])

        t_crit = stats.t.ppf((1 + confidence) / 2, df=n_obs - 1)
        sems   = np.array([v.std(ddof=1) / np.sqrt(len(v)) for v in vals])
        ci_lo  = means - t_crit * sems
        ci_hi  = means + t_crit * sems

        ax.fill_between(p_values, ci_lo, ci_hi,
                        alpha=0.20, color="steelblue",
                        label=f"{int(confidence*100)}% CI")
        ax.plot(p_values, means, marker="o", lw=2,
                color="steelblue", label="mean")
        ax.axhline(means[0], color="gray", lw=1, ls="--",
                   label=f"baseline p=0: {means[0]:.3f} s")

        ax.set_title(mode.replace("_", " "), fontsize=13, fontweight="bold")
        ax.set_xlabel("perturbation  p")
        ax.set_ylabel(r"$\mathbb{E}[t_{\mathrm{div}}]$ (s)")
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=9)

    fig.suptitle(
        r"$\mathbb{E}_{R}\!\left[\mathbb{E}_{\mathrm{runs}}"
        r"\!\left[t_{\mathrm{div}}(f_p(R))\right]\right]$"
        f" vs perturbation $p$\n"
        f"(ε={epsilon},  n_runs={n_runs},  n_files={n_files},"
        f"  t_train={t_training} s,  {int(confidence*100)}% CI)\n"
        f"[Fixed spectral scaling · Sparse W_in 20%]",
        fontsize=11,
    )
    plt.tight_layout()
    return fig


## Run the experiment

In [ ]:
# ── Data setup ────────────────────────────────────────────────────────────────

DATA_DIR   = Path("../repeated_10_scale_125")
file_paths = sorted(DATA_DIR.glob("*.graphml"))
print(f"Found {len(file_paths)} connectome files")

G0 = nx.read_graphml(file_paths[0])
print(f"First file — Nodes: {G0.number_of_nodes()}, Edges: {G0.number_of_edges()}")


In [ ]:
# ── Run experiment ────────────────────────────────────────────────────────────
# 20 files × 5 runs = 100 samples per (mode, p) point.
# Increase N_FILES for more statistical power (at the cost of runtime).

N_FILES  = 20
N_RUNS   = 5
P_VALUES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

results = run_experiment(
    file_paths[:N_FILES],
    p_values           = P_VALUES,
    n_runs             = N_RUNS,
    perturbation_types = PERTURBATION_TYPES,
    t_thermalization   = 30,
    t_training         = 200,
    t_prediction       = 50,
    J                  = 0.9,
    lambda_reg         = 3e-6,
    sigma              = 10.0,
    rho                = 28.0,
    beta               = 8 / 3,
    dt                 = 0.01,
    input_scale        = 0.5,
    alpha              = 0.3,
    sparse_input_frac  = 0.2,   # 20 % of nodes receive Lorenz input
    epsilon            = 1.0,
    seed_base          = 42,
)


In [ ]:
# ── Plot results ──────────────────────────────────────────────────────────────

fig = plot_experiment(
    results,
    p_values           = P_VALUES,
    epsilon            = 1.0,
    n_runs             = N_RUNS,
    n_files            = N_FILES,
    t_training         = 200,
    perturbation_types = PERTURBATION_TYPES,
    confidence         = 0.95,
)
plt.show()
